In [16]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

In [17]:
from datasets import load_dataset
dataset_path = "dataset/dataset_curation_agent/valid_codes.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset = dataset.shuffle(seed=42)

total_len = len(dataset)
train_end = int(0.7 * total_len)
eval_end = int(0.9 * total_len)

# Slice the dataset
train_dataset = dataset.select(range(0, train_end))
eval_dataset  = dataset.select(range(train_end, eval_end))
test_dataset  = dataset.select(range(eval_end, total_len))

# Print lengths to verify
print("Total:", total_len)
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))
print("\n")

print("-------------------------------------\nTrain:",len(train_dataset),"example: ",train_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nEval:", len(eval_dataset),"\n example: ",eval_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nTest:", len(test_dataset),"\n example: ",test_dataset[0],"\n-------------------------------------\n")

Generating train split: 0 examples [00:00, ? examples/s]

Total: 654
Train: 457
Eval: 131
Test: 66


-------------------------------------
Train: 457 example:  {'task': 'Fix the issue in the following Python code.', 'buggy_code': 'def _u_in(self, u):\n    return u >= 0.0 or u <= 1.0', 'correct_code': 'def _u_in(self, u):\n    return u >= 0.0 and u <= 1.0', 'unit_test': 'def check(candidate):\n    # Test cases for numbers within the range [0.0, 1.0]\n    assert candidate(0.0) == True\n    assert candidate(1.0) == True\n    assert candidate(0.5) == True\n    \n    # Test cases for numbers outside the range\n    assert candidate(-0.1) == False\n    assert candidate(1.1) == False\n    \n    # Edge case: exactly at the boundaries\n    assert candidate(0.0) == True  # Lower boundary\n    assert candidate(1.0) == True  # Upper boundary\n\n    # Test cases for numbers equal to the boundaries\n    assert candidate(-0.0) == True  # -0.0 is equivalent to 0.0 in Python\n    \n    # Additional test case with a number very close to the boundaries\n    asse

In [18]:
EVAL_REFERENCES = [ex["correct_code"] for ex in eval_dataset]
TEST_REFERENCES = [ex["correct_code"] for ex in test_dataset]
print("eval_references:", EVAL_REFERENCES[0],"\n")
print("test_references:", TEST_REFERENCES[0],"\n")

eval_references: def GetToolchainEnv(self, additional_settings=None):
  """Returns the variables toolchain would set for build steps."""
  env = self.GetSortedXcodeEnv(additional_settings=additional_settings)
  if self.flavor == 'win':
    env = self.GetMsvsToolchainEnv(
        additional_settings=additional_settings)
  return env 

test_references: def get_prep_value(self, value):
    if value is not None:
        return int(value)
    return super(SaneTimeField,self).get_prep_value(value) 



In [19]:
# Just for your train split
def formatting_prompts_func(examples):
    output_text = []
    for i in range(len(examples["task"])):
        task = examples["task"][i]
        buggy_code = examples["buggy_code"][i]
        correct_code = examples["correct_code"][i]

        if buggy_code.strip():
            text = f"""### Instruction:
            {task}

            ### Buggy Code:
            {buggy_code}

            ### Fixed Code:
            {correct_code}
            """
            output_text.append(text)
    return output_text

In [20]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [21]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
        max_seq_length = 512,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.5.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [23]:
import neptune
run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)



[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-132


In [24]:
import evaluate
from codebleu import compute_codebleu
# Metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")
#code_eval = evaluate.load("code_eval")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # Mask handling
    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id

    # Decode
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    decoded_completions = []
    for pred in decoded_preds:
        parts = pred.split("### Fixed Code:")
        completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
        decoded_completions.append(completion)

    # Standard metrics
    bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])
    refs = [[ref] for ref in EVAL_REFERENCES]
    codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

    # Simulate pass@1: just one candidate per sample
    #predictions_for_code_eval = [[c] for c in tqdm(decoded_completions, desc="Preparing predictions")]


    # print("Computing pass@k...")
    # pass_at_k, _ = code_eval.compute(
    #     references=EVAL_REFERENCES,
    #     predictions=predictions_for_code_eval,
    #     k=[1],
    # )
    # print("pass@k")
    return {
        #"pass@1": pass_at_k["pass@1"],
        "codebleu": codebleu_scores["codebleu"],
        **bleu_score,
        **rouge_score,
        **accuracy
    }

In [25]:
# from trl import SFTTrainer, SFTConfig
# import time
# start=time.time()
#
# # SFT Config
# config = SFTConfig(
#     dataset_num_proc = 1,
#     #dataset_text_field="prompt",#Depends on the colum of your data set
#     learning_rate=2e-4,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=1,
#     num_train_epochs=5,
#     report_to="none",
#     logging_steps=100,
#     #max_steps=500,
#     #eval_accumulation_steps=100,
# )
# trainer = SFTTrainer(
#     model=model,  # base or PEFT model
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     formatting_func=formatting_prompts_func,
#     args=config,
#     warmup_steps = 5,
#     weight_decay = 0.01,
#     compute_metrics = compute_metrics,
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )
# # metrics = trainer.evaluate()
# # print("Metrics:",metrics)
# trainer.train()
#
# end = time.time()
# length = end - start
#
# hours = int(length // 3600)
# minutes = int((length % 3600) // 60)
# seconds = int(length % 60)
#
# print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")
#

In [26]:
# torch.cuda.empty_cache()
# metrics = trainer.evaluate()
# print("Metrics:",metrics)


In [27]:
# from tqdm import tqdm
# import evaluate
#
# def evaluate_pass_at_k(model, tokenizer, prompts, references, k_values=[1, 5, 10], num_completions=10, max_new_tokens=256):
#     code_eval = evaluate.load("code_eval")
#
#     all_predictions = []
#
#     model.eval()
#     for prompt in tqdm(prompts, desc="Generating Completions"):
#         input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
#         outputs = model.generate(
#             input_ids=input_ids,
#             do_sample=True,
#             top_k=50,
#             top_p=0.95,
#             temperature=0.7,
#             num_return_sequences=num_completions,
#             max_new_tokens=max_new_tokens,
#             pad_token_id=tokenizer.eos_token_id
#         )
#         torch.cuda.empty_cache()
#         decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
#         #print(f"decoded_completions: {decoded}")
#         # Extract completions
#         cleaned = []
#         for d in decoded:
#             parts = d.split("### Fixed Code:")
#             cleaned.append(parts[-1].strip() if len(parts) > 1 else d.strip())
#
#         all_predictions.append(cleaned)
#
#     print("\n✅ All completions generated. Computing pass@k...\n")
#     result, _ = code_eval.compute(
#         references=references,
#         predictions=all_predictions,
#         k=k_values,
#     )
#
#     print("🎯 Final pass@k scores:")
#     for k in k_values:
#         score = result.get(f'pass@{k}', 'N/A')
#         if isinstance(score, (float, int)):
#             print(f"pass@{k}: {score:.4f}")
#         else:
#             print(f"pass@{k}: {score}")


In [28]:
# prompts = []
# for ex in test_dataset:
#     prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
#
# ### Instruction:
# {ex["task"]}
#
# ### Input:
# {ex["buggy_code"]}
#
# ### Response:"""
#     prompts.append(prompt)
#
# pass_at_k_scores = evaluate_pass_at_k(model, tokenizer, prompts, TEST_REFERENCES)
# print(pass_at_k_scores)

In [29]:
from trl import SFTTrainer, SFTConfig
import time
from tqdm import tqdm

def objective(trial):
    start=time.time()
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5,5e-4, log=True)
    num_epochs = trial.suggest_int("num_train_epochs", 5, 10)
    batch_size=2

    total_examples = len(train_dataset)
    steps_per_epoch = total_examples // batch_size
    max_steps = num_epochs * steps_per_epoch
    #max_steps=200

    # SFT Config
    config = SFTConfig(
    dataset_num_proc = 1,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=batch_size,
    num_train_epochs=num_epochs,
    logging_steps=100,
    report_to="none",
    max_steps=max_steps,
    eval_accumulation_steps=100,
    )
    trainer = SFTTrainer(
        model=model,  # base or PEFT model

        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        formatting_func=formatting_prompts_func,
        args=config,
        warmup_steps = 5,
        weight_decay = 0.01,
        compute_metrics = compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
    trainer.train()

    # # === Log training loss to Neptune ===
    # for record in trainer.state.log_history:
    #     if "loss" in record:
    #         step = record.get("step", None)
    #         loss = record["loss"]
    #         run[f"optuna/trial/{trial.number}/train/loss"].append({"step": step, "value": loss})

    # === Evaluate model ===
    metrics = trainer.evaluate()
    print("Metrics:", metrics)

    # === Log evaluation and hyperparams ===
    run[f"optuna/trial/{trial.number}/metrics"] = metrics
    run[f"optuna/trial/{trial.number}/params"] = {
        "learning_rate": learning_rate,
        "num_epochs": num_epochs
    }

    # === Duration tracking ===
    end = time.time()
    length = end - start
    h, m, s = int(length // 3600), int((length % 3600) // 60), int(length % 60)
    print(f"⏱️ It took {h}h {m}m {s}s to train the model!")

    # === Return metric to minimize ===
    return metrics["eval_loss"]  # Or any other objective


In [30]:
import optuna
import neptune.integrations.optuna as optuna_utils
start=time.time()
neptune_callback = optuna_utils.NeptuneCallback(run=run)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5, callbacks=[neptune_callback], show_progress_bar=True)
end = time.time()
length = end - start

hours = int(length // 3600)
minutes = int((length % 3600) // 60)
seconds = int(length % 60)
print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

[I 2025-05-28 22:19:30,183] A new study created in memory with name: no-name-090fc612-fd68-4639-9a89-f1e12b251cba


  0%|          | 0/5 [00:00<?, ?it/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/457 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/131 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 457 | Num Epochs = 28 | Total steps = 1,596
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


...

Unsloth: Tokenizing ["text"]:   0%|          | 0/131 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 457 | Num Epochs = 28 | Total steps = 1,596
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


Step,Training Loss
100,0.064600
200,0.087300
300,0.078100
400,0.153500
500,0.085500
600,0.049200
700,0.039500
800,0.032900
900,0.030700
1000,0.030300


[W 2025-05-28 23:51:02,142] Trial 1 failed with parameters: {'learning_rate': 0.0003596425942318091, 'num_train_epochs': 7} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_609857/2185944450.py", line 42, in objective
    trainer.train()
  File "/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 314, in _fast_inner_training_loop
  File "<string>", line 77, in _unsloth_training_step
  File "/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-packages/accelerate/accelerator.py", line 2473, in backward
 

KeyboardInterrupt: 

In [32]:
# Get the best parameters
best_trial = study.best_trial

best_params = best_trial.params
print("best_params: ",best_params)

best_value = best_trial.value
print("Eval loss:", best_value)
run.stop()